# Relativistic string modes and T-duality

*cosmology · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/string_modes)

The Nambu–Goto string in light-cone gauge is 24 free transverse waves. One batched mode sum draws a vibrating closed or open string; the energy in its modes is the level $N$, quantised as $\alpha' M^2 = N - 1$ (open) or $4(N-1)$ (closed), with $1, 24, 324, 3200, \dots$ states read off the power series $\prod (1-q^n)^{-24}$. On a circle of radius $R$ the closed-string spectrum is unchanged by $R \leftrightarrow \alpha'/R$: T-duality. String theory is a framework with no direct experimental confirmation.

**Vector form**

$$(\partial_\tau^2 - \partial_\sigma^2)\,X^i = 0,\qquad X^i = \sqrt{2\alpha'}\sum_{n>0}\frac{1}{n}\,\mathrm{Re}\big[i\alpha_n^i e^{-in(\tau-\sigma)} + i\tilde\alpha_n^i e^{-in(\tau+\sigma)}\big],\qquad \alpha' M^2 = 2(N + \tilde N - 2),\qquad \sum_N d_N q^N = \prod_{n\ge1}(1-q^n)^{-24},\qquad M^2 = \frac{n^2}{R^2} + \frac{w^2R^2}{\alpha'^2} + \frac{2}{\alpha'}(N + \tilde N - 2)$$

**Index (tensor) form**

$$X_{tsi} = \sqrt{2\alpha'}\,\mathrm{Re}\Big[E^{(R)}_{tsn}\,\tfrac{i}{n}\,\alpha_{ni} + E^{(L)}_{tsn}\,\tfrac{i}{n}\,\tilde\alpha_{ni}\Big],\qquad N + \tilde N = \frac{1}{4\pi\alpha'}\int\!\big(\dot X_{si}\dot X_{si} + X'_{si}X'_{si}\big)\,d\sigma,\qquad d_N = \big[e^{\,24\,T}\big]_{N0},\quad T_{ij} = \tfrac{\sigma_1(i-j)}{i-j}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\alpha_{ni},\ \tilde\alpha_{ni}$ | `oscillators(n_a, n_b, kind)` | (N_modes, 3) | complex amplitudes of the right- and left-moving modes $n$ in 3 of the 24 transverse directions $i$ |
| $E^{(R)}_{tsn}$ | `torch.polar(1, -n*(tau - sigma))` | (T, S, N_modes) | phase $e^{-in(\tau-\sigma)}$ of every mode at every worldsheet point (broadcast) |
| $X_{tsi}$ | `einsum('tsn,nd->tsd', 1j*E/n, alpha)` | (T, S, 3) | repeated mode index $n$ ⇒ summed: the string shape at every $\tau$ and $\sigma$ |
| $\dot X_{si}\dot X_{si}$ | `einsum('tsi,tsi->ts', X_dot, X_dot)` | (T, S) | energy density; integrated over $\sigma$ it gives the level $N$ (Parseval) |
| $\sigma_1(m)$ | `(m[:, None] % m == 0) @ m` | (M,) | divisor sums from a broadcast divisibility mask: $\log\prod(1-q^n)^{-1} = \sum_m \sigma_1(m) q^m/m$ |
| $d_N$ | `matrix_exp(toeplitz)[:, 0]` | (N_max+1,) | truncated power series multiply like lower-triangular Toeplitz matrices, so $\exp$ of the series is a matrix exponential |
| $M^2_{bs}$ | `circle_mass_squared(R[:, None], n, w, N, N_t)` | (B, S) | masses of all level-matched states $N-\tilde N = nw$ on every radius $R_b$ at once |

Units $\alpha' = 1$. Status: string theory is a mathematically consistent candidate framework for quantum gravity, but there is no direct experimental evidence for it; nothing here is a prediction that has been tested. What is shown is standard textbook mathematics of the bosonic string (Goddard–Goldstone–Rebbi–Thorn 1973; Polchinski, String Theory vol. 1): the light-cone quantisation is consistent only in $D = 26$, with intercept $a = 1$, which leaves a tachyon at $N = 0$ ($M^2 < 0$, red), a massless vector at $N = 1$ for open strings and the graviton, B-field and dilaton at $N = \tilde N = 1$ for closed strings. Superstrings (D = 10) remove the tachyon. The drawn string is a classical configuration whose oscillator energy equals the level $N$ (each quantum of mode $n$ carries $n$ units); for closed strings the right and left movers are chosen with $N = \tilde N$ (level matching, $\int \dot X\cdot X'\,d\sigma = 0$), and the string is in its rest frame, so the light-cone coordinate $X^-$ is not drawn. The T-duality panel follows the masses of a few states as the radius changes: momentum states ($n/R$) and winding states ($wR/\alpha'$) swap under $R \to \alpha'/R$, and at the self-dual radius $R = \sqrt{\alpha'}$ extra states become massless. The title checks that the whole truncated spectrum, with multiplicities, is identical at $R$ and $\alpha'/R$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/string_modes/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.plotting import frame_indices
from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    kind, n_a, n_b = params["string"], params["mode_a"], params["mode_b"]
    alpha, alpha_t = (a.to(dev) for a in tensors.oscillators(n_a, n_b, kind))
    sigma = tensors.sigma_grid(params["n_points"], kind, dev)
    steps = frame_indices(72, 72)
    tau = tensors.tau_grid(72, dev)[steps]  # one worldsheet time per frame

    sheet = solve.worldsheet(tau, sigma, alpha, alpha_t, kind)  # X: (F, S, 3)
    level = round(sheet["N"])
    print(f"{kind} string: modes {n_a} and {n_b}, oscillator sum N = {sheet['N']:.3f}"
          + (f", Ñ = {sheet['N_t']:.3f} (level matched)" if kind == "closed" else ""))  # fmt: skip
    lf = sheet["level_field"]
    print(f"level from the field energy (1/4πα′)∫(Ẋ²+X′²)dσ: {lf.min():.6f} … {lf.max():.6f} over τ (conserved)")

    spec = solve.spectrum(kind, params["n_levels"])
    print("level N:        " + "  ".join(f"{int(n):>9d}" for n in spec["N"]))
    print("α′M²:           " + "  ".join(f"{m:>9.0f}" for m in spec["M2"]))
    print("degeneracy:     " + "  ".join(f"{d:>9.4g}" for d in spec["degeneracy"]))
    m2_state = float(spec["M2"][level]) if level < len(spec["M2"]) else float("nan")
    print(f"this classical configuration sits at level N = {level}: α′M² = {m2_state:.0f} (quantum, intercept a = 1)")

    radii = tensors.radii(1 / params["R_max"], params["R_max"], len(steps), dev)
    M2, _ = solve.circle_spectrum(torch.cat([radii, 1 / radii]), k_max=6, n_max=3)  # (2F, S) all states, all radii
    sorted_M2 = M2.sort(dim=1).values
    gap = (sorted_M2[: len(radii)] - sorted_M2[len(radii) :]).abs().max(dim=1).values  # (F,)
    print(f"T-duality R ↔ α′/R on {M2.shape[1]} states × {len(radii)} radii: max |ΔM²| = {gap.max():.2e}")

    curves = (solve.duality_curves(radii), solve.duality_curves(1 / radii))
    level = min(level, params["n_levels"])
    return plot.animate(sheet, kind, spec, level, radii, curves, list(solve.NAMED_STATES), gap,
                        out_dir / "string_modes.gif")  # fmt: skip


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/string_modes/equation.py
r"""Physics → tensor translation: the bosonic string in light-cone gauge.

In light-cone gauge the Nambu–Goto string reduces to D − 2 = 24 free transverse fields Xⁱ(τ, σ) obeying the wave
equation (∂τ² − ∂σ²) Xⁱ = 0. In the centre-of-mass frame (p = 0, x = 0), with α′ the Regge slope:

  closed (σ ∈ [0, 2π)):  Xⁱ = √(2α′) Σ_{n>0} (1/n) Re[ i α_nⁱ e^{−in(τ−σ)} + i α̃_nⁱ e^{−in(τ+σ)} ]
  open   (σ ∈ [0, π]):   Xⁱ = 2√(2α′) Σ_{n>0} (1/n) Re[ i α_nⁱ e^{−inτ} ] cos nσ

Both are batched mode sums: a phase tensor E_{tsn} (time t, point s, mode n) contracted with the amplitudes α_{nd}
over the repeated mode index n (einsum 'tsn,nd->tsd'). The oscillator level is the energy in the modes,

  N = Σ_{n>0} α_{−n}·α_n = Σ_n |α_n|²   (quantum: Σ n k_n with k_n quanta in mode n)
    = (1/4πα′) ∫ (Ẋ² + X′²) dσ          (Parseval; closed strings give N + Ñ)

and the quantum mass formulae (D = 26, normal-ordering intercept a = 1) are

  open:  α′M² = N − 1            closed:  α′M² = 2(N + Ñ − 2) = 4(N − 1),  with level matching N = Ñ.

The number of states at level N is the coefficient of qᴺ in ∏_{n≥1} (1 − qⁿ)^{−24} (24 colours of oscillator).
"""

import math

import torch

from .tensors import ALPHA_PRIME


def _phases(tau: torch.Tensor, sigma: torch.Tensor, n: torch.Tensor, sign: int) -> torch.Tensor:
    r"""E_{tsn} = e^{−in(τ_t − sign·σ_s)}: right movers (sign +1) or left movers (sign −1).   → (T, S, N) complex"""
    arg = n[None, None, :] * (tau[:, None, None] - sign * sigma[None, :, None])
    return torch.polar(torch.ones_like(arg), -arg)


def _modes(alpha: torch.Tensor) -> torch.Tensor:
    return torch.arange(1, alpha.shape[0] + 1, dtype=alpha.real.dtype, device=alpha.device)  # n = 1 … N_modes


def embedding(tau, sigma, alpha, alpha_t, kind):
    r"""Xⁱ(τ, σ) for every time and point at once.   tau (T,), sigma (S,), alpha (N, 3) complex → X (T, S, 3)

    The mode index n is summed in the einsum; division by n weights high modes down (their energy n|·|² is equal).
    """
    n = _modes(alpha)
    if kind == "closed":
        X = torch.einsum("tsn,nd->tsd", 1j * _phases(tau, sigma, n, +1) / n, alpha)
        X = X + torch.einsum("tsn,nd->tsd", 1j * _phases(tau, sigma, n, -1) / n, alpha_t)
        return math.sqrt(2 * ALPHA_PRIME) * X.real
    e = torch.polar(torch.ones_like(tau[:, None] * n), -tau[:, None] * n)  # (T, N) e^{−inτ}
    c = torch.cos(sigma[:, None] * n).to(e.dtype)  # (S, N) cos nσ
    return 2 * math.sqrt(2 * ALPHA_PRIME) * torch.einsum("tn,sn,nd->tsd", 1j * e / n, c, alpha).real


def derivatives(tau, sigma, alpha, alpha_t, kind):
    """(Ẋ, X′), both (T, S, 3): the same mode sums differentiated term by term."""
    n = _modes(alpha)
    k = math.sqrt(2 * ALPHA_PRIME)
    if kind == "closed":
        right = torch.einsum("tsn,nd->tsd", _phases(tau, sigma, n, +1), alpha).real  # ∂X_R/∂τ
        left = torch.einsum("tsn,nd->tsd", _phases(tau, sigma, n, -1), alpha_t).real
        return k * (right + left), k * (left - right)  # X′ = ∂σ: right movers flip sign
    e = torch.polar(torch.ones_like(tau[:, None] * n), -tau[:, None] * n)  # (T, N)
    c = torch.cos(sigma[:, None] * n).to(e.dtype)
    s = torch.sin(sigma[:, None] * n).to(e.dtype)
    X_dot = 2 * k * torch.einsum("tn,sn,nd->tsd", e, c, alpha).real
    X_prime = -2 * k * torch.einsum("tn,sn,nd->tsd", 1j * e, s, alpha).real
    return X_dot, X_prime


def _integrate(f: torch.Tensor, sigma: torch.Tensor, kind: str) -> torch.Tensor:
    """∫ f dσ over the string, f (T, S) → (T,): rectangle rule on the closed loop, trapezoid on [0, π]."""
    if kind == "closed":
        return f.sum(-1) * (2 * math.pi / len(sigma))
    return torch.trapezoid(f, sigma, dim=-1)


def level_from_field(X_dot, X_prime, sigma, kind):
    r"""N (+ Ñ) = (1/4πα′) ∫ (Ẋⁱ Ẋⁱ + X′ⁱ X′ⁱ) dσ, i summed.   (T, S, 3) → (T,)"""
    density = torch.einsum("tsi,tsi->ts", X_dot, X_dot) + torch.einsum("tsi,tsi->ts", X_prime, X_prime)
    return _integrate(density, sigma, kind) / (4 * math.pi * ALPHA_PRIME)


def sigma_momentum(X_dot, X_prime, sigma):
    r"""P_σ = ∫ Ẋⁱ X′ⁱ dσ = 2πα′ (Ñ − N): generates σ translations; level matching says it vanishes.   → (T,)"""
    return _integrate(torch.einsum("tsi,tsi->ts", X_dot, X_prime), sigma, "closed")


def oscillator_level(alpha: torch.Tensor) -> float:
    r"""N = Σ_{n,i} |α_nⁱ|² (the classical value of Σ α_{−n}·α_n)."""
    return float((alpha.abs() ** 2).sum())


def level_degeneracies(n_max: int, colours: int = 24) -> torch.Tensor:
    r"""Coefficients d_0 … d_{n_max} of ∏_{n≥1} (1 − qⁿ)^{−colours}, as a tensor power series.

    log ∏ (1 − qⁿ)^{−c} = c Σ_m σ₁(m)/m qᵐ with σ₁(m) the sum of the divisors of m (a broadcast divisibility mask).
    Truncated power series multiply like lower-triangular Toeplitz matrices, so exp of the series is matrix_exp.
    """
    m = torch.arange(1, n_max + 1, dtype=torch.float64)
    divides = (m[:, None] % m[None, :] == 0).to(torch.float64)  # (M, M): d | m
    sigma1 = divides @ m  # (M,) divisor sums
    log_series = torch.cat([torch.zeros(1, dtype=torch.float64), colours * sigma1 / m])  # (n_max + 1,)
    i = torch.arange(n_max + 1)
    lag = i[:, None] - i[None, :]  # Toeplitz index i − j
    toeplitz = torch.where(lag >= 0, log_series[lag.clamp(min=0)], torch.zeros(()))
    return torch.linalg.matrix_exp(toeplitz)[:, 0]


def open_mass_squared(N: torch.Tensor) -> torch.Tensor:
    """α′M² = N − 1 in units of 1/α′ (intercept a = 1 in D = 26)."""
    return (N - 1) / ALPHA_PRIME


def closed_mass_squared(N: torch.Tensor) -> torch.Tensor:
    """α′M² = 2(N + Ñ − 2) = 4(N − 1) for level-matched states N = Ñ."""
    return 4 * (N - 1) / ALPHA_PRIME


def circle_mass_squared(R, n, w, N, N_t):
    r"""Closed string with one dimension on a circle of radius R: momentum n/R, winding wR/α′.

    M² = (n/R)² + (wR/α′)² + (2/α′)(N + Ñ − 2), with level matching N − Ñ = n w. All arguments broadcast.
    Swapping R ↔ α′/R and n ↔ w leaves M² unchanged: T-duality.
    """
    return (n / R) ** 2 + (w * R / ALPHA_PRIME) ** 2 + 2 * (N + N_t - 2) / ALPHA_PRIME


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/string_modes/tensors.py
"""Build the concrete tensors: worldsheet grids and the oscillator amplitudes α_n, α̃_n of a chosen state.

Units: α′ = 1 (string length √α′ = 1). Only three of the 24 transverse directions are excited, so the string can be
drawn in 3D; the other 21 sit at zero.
"""

import math

import torch

ALPHA_PRIME = 1.0  # Regge slope α′ (sets the string tension T = 1/(2πα′))
TRANSVERSE = 24  # D − 2 transverse directions in the critical dimension D = 26


def sigma_grid(n: int, kind: str, device: torch.device) -> torch.Tensor:
    """Points along the string: [0, 2π) for a closed loop (periodic), [0, π] for an open string.   (S,)"""
    if kind == "closed":
        return torch.arange(n, dtype=torch.float64, device=device) * (2 * math.pi / n)
    return torch.linspace(0, math.pi, n, dtype=torch.float64, device=device)


def tau_grid(n: int, device: torch.device) -> torch.Tensor:
    """One full period of worldsheet time, τ ∈ [0, 2π].   (T,)"""
    return torch.linspace(0, 2 * math.pi, n, dtype=torch.float64, device=device)


def oscillators(n_a: int, n_b: int, kind: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Amplitudes (α_{nd}, α̃_{nd}), each (max(n_a, n_b), 3) complex.

    closed: right movers in mode n_a, circularly polarised in the (1, 2) plane, and left movers in mode n_b in the
            (2, 3) plane. Each side gets level N = lcm(n_a, n_b), i.e. N / n quanta of mode n, so N = Ñ
            (level matching).
    open:   one quantum in mode n_a (|α|² = n_a) plus one in mode n_b (|α|² = n_b): N = n_a + n_b. α̃ = 0.
    """
    modes = max(n_a, n_b)
    alpha = torch.zeros(modes, 3, dtype=torch.complex128)
    alpha_t = torch.zeros(modes, 3, dtype=torch.complex128)
    plane_a = torch.tensor([1, 1j, 0], dtype=torch.complex128) / math.sqrt(2)  # unit circular polarisation
    plane_b = torch.tensor([0, 1, 1j], dtype=torch.complex128) / math.sqrt(2)
    if kind == "closed":
        level = math.lcm(n_a, n_b)
        alpha[n_a - 1] = math.sqrt(level) * plane_a
        alpha_t[n_b - 1] = math.sqrt(level) * plane_b
    else:
        alpha[n_a - 1] += math.sqrt(n_a) * plane_a
        alpha[n_b - 1] += math.sqrt(n_b) * plane_b
    return alpha, alpha_t


def radii(r_min: float, r_max: float, n: int, device: torch.device) -> torch.Tensor:
    """Log-spaced compactification radii R (units of √α′), symmetric about the self-dual radius when r_min = 1/r_max."""
    return torch.logspace(math.log10(r_min), math.log10(r_max), n, dtype=torch.float64, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/string_modes/solve.py
"""Evaluate the string on the whole worldsheet at once, the level degeneracies and the circle spectrum.

No time stepping: the light-cone solution is known in closed form, so every (τ, σ) point is one mode sum.
"""

import torch

from .equation import (
    circle_mass_squared,
    closed_mass_squared,
    derivatives,
    embedding,
    level_degeneracies,
    level_from_field,
    open_mass_squared,
    oscillator_level,
)


def worldsheet(tau, sigma, alpha, alpha_t, kind: str) -> dict:
    """X (T, S, 3) plus the level measured from the field energy at every τ (T,)."""
    X = embedding(tau, sigma, alpha, alpha_t, kind)
    X_dot, X_prime = derivatives(tau, sigma, alpha, alpha_t, kind)
    return {"X": X, "level_field": level_from_field(X_dot, X_prime, sigma, kind),
            "N": oscillator_level(alpha), "N_t": oscillator_level(alpha_t)}  # fmt: skip


def spectrum(kind: str, n_max: int) -> dict:
    """Mass levels α′M² and their degeneracies for N = 0 … n_max (closed strings: d_N² from both sides)."""
    N = torch.arange(n_max + 1, dtype=torch.float64)
    d = level_degeneracies(n_max)
    if kind == "closed":
        return {"N": N, "M2": closed_mass_squared(N), "degeneracy": d**2}
    return {"N": N, "M2": open_mass_squared(N), "degeneracy": d}


def circle_spectrum(radii: torch.Tensor, k_max: int, n_max: int) -> tuple[torch.Tensor, torch.Tensor]:
    """All level-matched closed-string states with |n|, |w| ≤ k_max and N, Ñ ≤ n_max, for every radius at once.

    Broadcast grid (R, n, w, N, Ñ) → keep the entries with N − Ñ = n w → M² (B, S) and multiplicity d_N d_Ñ (S,).
    """
    k = torch.arange(-k_max, k_max + 1, dtype=torch.float64, device=radii.device)
    lvl = torch.arange(n_max + 1, dtype=torch.float64, device=radii.device)
    n, w, N, N_t = torch.meshgrid(k, k, lvl, lvl, indexing="ij")  # each (K, K, L, L)
    matched = N - N_t == n * w  # level matching
    d = level_degeneracies(n_max).to(radii.device)
    weight = (d[:, None] * d[None, :]).expand_as(N)[matched]  # (S,)
    M2 = circle_mass_squared(radii[:, None], n[matched], w[matched], N[matched], N_t[matched])  # (B, S)
    return M2, weight


# a few low-lying states (n, w, N, Ñ) to draw as curves M²(R): all obey N − Ñ = n w
NAMED_STATES = {
    "tachyon (0,0,0,0)": (0, 0, 0, 0),
    "graviton etc. (0,0,1,1)": (0, 0, 1, 1),
    "momentum n=1": (1, 0, 1, 1),
    "winding w=1": (0, 1, 1, 1),
    "n=w=1, N=1, Ñ=0": (1, 1, 1, 0),
}


def duality_curves(radii: torch.Tensor) -> torch.Tensor:
    """α′M²(R) of the named states, (B, 5): one broadcast over radii and states."""
    s = torch.tensor(list(NAMED_STATES.values()), dtype=torch.float64, device=radii.device)  # (5, 4)
    return circle_mass_squared(radii[:, None], s[:, 0], s[:, 1], s[:, 2], s[:, 3])


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/string_modes/plot.py
"""Animate the vibrating string, its level in the mass spectrum, and T-duality of the circle spectrum."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, save_gif


def animate(sheet: dict, kind: str, spec: dict, level: int, radii, curves, names, duality_gap, path: Path) -> Path:
    """sheet["X"]: (F, S, 3) one string shape per frame; radii (F,), curves (F, 5) α′M²(R) and (F, 5) at α′/R."""
    X = sheet["X"].cpu().numpy()
    if kind == "closed":
        X = np.concatenate([X, X[:, :1]], axis=1)  # close the loop for drawing
    R = radii.cpu().numpy()
    M2_R, M2_dual = (c.cpu().numpy() for c in curves)
    gap = duality_gap.cpu().numpy()
    M2_lvl, deg = spec["M2"].numpy(), spec["degeneracy"].numpy()

    fig = plt.figure(figsize=(10.5, 3.9))
    ax = fig.add_subplot(1, 3, 1, projection="3d")
    ax_s = fig.add_subplot(1, 3, 2)
    ax_t = fig.add_subplot(1, 3, 3)

    reach = 1.05 * np.abs(X).max()
    for set_lim in (ax.set_xlim, ax.set_ylim, ax.set_zlim):
        set_lim(-reach, reach)
    ax.set_xlabel("$X^1$", labelpad=-8)
    ax.set_ylabel("$X^2$", labelpad=-8)
    ax.set_zlabel("$X^3$", labelpad=-8)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])
    line = ax.plot([], [], [], "-", lw=2, color=ACCENT[0])[0]
    dot = ax.plot([], [], [], "o", ms=5, color=ACCENT[1])[0]
    shadow = ax.plot([], [], [], "-", lw=1, color="#9ca3af")[0]
    title = ax.set_title("", fontsize=8)
    lvl_name = "N + Ñ" if kind == "closed" else "N"

    # --- spectrum: log10 degeneracy at each mass level ---
    colours = ["#dc2626" if m < 0 else "#93c5fd" for m in M2_lvl]
    colours[level] = ACCENT[0] if M2_lvl[level] >= 0 else "#dc2626"
    width = 0.7 * (M2_lvl[1] - M2_lvl[0])
    bars = ax_s.bar(M2_lvl, np.log10(deg) + 0.4, bottom=-0.4, width=width, color=colours, edgecolor="#1e40af", lw=0.5)
    bars[level].set_edgecolor("#111827")
    bars[level].set_linewidth(2)
    for m, d in zip(M2_lvl, deg):
        ax_s.text(m, np.log10(d) + 0.15, f"{d:.3g}" if d < 1e5 else f"{d:.1e}", ha="center", fontsize=6)
    ax_s.set_xlabel("$\\alpha' M^2$" + ("  ($=4(N-1)$)" if kind == "closed" else "  ($=N-1$)"))
    ax_s.set_ylabel("$\\log_{10}$ degeneracy")
    ax_s.set_ylim(-0.4, np.log10(deg).max() * 1.15 + 0.5)
    ax_s.set_title(f"{kind} string levels from $\\prod(1-q^n)^{{-24}}$", fontsize=9)
    ax_s.text(M2_lvl[0], 0.9, "tachyon", color="#dc2626", ha="center", fontsize=7, rotation=90)
    marker = ax_s.text(0.97, 0.95, "", transform=ax_s.transAxes, ha="right", va="top", fontsize=8)

    # --- T-duality: α′M²(R) on a log axis, mirror image about R = √α′ ---
    grid = np.logspace(np.log10(R.min()), np.log10(R.max()), 300)
    s = np.array([(0, 0, 0, 0), (0, 0, 1, 1), (1, 0, 1, 1), (0, 1, 1, 1), (1, 1, 1, 0)], float)
    full = (s[:, 0] / grid[:, None]) ** 2 + (s[:, 1] * grid[:, None]) ** 2 + 2 * (s[:, 2] + s[:, 3] - 2)
    for j, name in enumerate(names):
        ax_t.plot(grid, full[:, j], color=ACCENT[j % len(ACCENT)], lw=1.3, label=name)
    ax_t.set_xscale("log")
    ax_t.set_ylim(-5, 17)
    ticks = [t for t in (1 / 5, 1 / 3, 1 / 2, 1, 2, 3, 5) if R.min() * 0.99 <= t <= R.max() * 1.01]
    ax_t.set_xticks(ticks, [f"{t:.2g}" for t in ticks])
    ax_t.xaxis.set_minor_formatter(plt.NullFormatter())
    ax_t.set_xlabel("circle radius $R/\\sqrt{\\alpha'}$")
    ax_t.set_ylabel("$\\alpha' M^2$")
    ax_t.axvline(1.0, color="#9ca3af", ls="--", lw=1)
    ax_t.legend(fontsize=6, loc="upper center", ncol=2)
    cur = ax_t.axvline(R[0], color="#111827", lw=1)
    cur_d = ax_t.axvline(1 / R[0], color="#f59e0b", lw=1)
    pts = ax_t.plot([], [], "o", ms=5, color="#111827", mfc="none")[0]
    pts_d = ax_t.plot([], [], "x", ms=5, color="#f59e0b")[0]
    t_title = ax_t.set_title("", fontsize=8)

    def update(f: int):
        line.set_data(X[f, :, 0], X[f, :, 1])
        line.set_3d_properties(X[f, :, 2])
        shadow.set_data(X[f, :, 0], X[f, :, 1])
        shadow.set_3d_properties(np.full(len(X[f]), -reach))
        dot.set_data([X[f, 0, 0]], [X[f, 0, 1]])
        dot.set_3d_properties([X[f, 0, 2]])
        tau = 2 * np.pi * f / max(1, len(X) - 1)
        title.set_text(f"τ = {tau:4.2f}   {lvl_name} from (1/4πα′)∫(Ẋ²+X′²)dσ ={float(sheet['level_field'][f]):.4f}")
        marker.set_text(f"this string: N = {level}" + ("" if kind == "open" else " = Ñ"))
        cur.set_xdata([R[f]])
        cur_d.set_xdata([1 / R[f]])
        pts.set_data(np.full(len(names), R[f]), M2_R[f])
        pts_d.set_data(np.full(len(names), 1 / R[f]), M2_dual[f])
        t_title.set_text(f"T-duality R={R[f]:.2f} ↔ α′/R={1 / R[f]:.2f}\nfull spectra agree: max |ΔM²| = {gap[f]:.1e}")

    fig.tight_layout()
    return save_gif(fig, update, list(range(len(X))), path, fps=15)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'string': 'closed',
    'mode_a': 2,
    'mode_b': 1,
    'n_points': 240,
    'n_levels': 6,
    'R_max': 3.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.string_modes")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.string_modes.main")
problem_dir = Path("problems/cosmology/string_modes")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
